Read full pharmaprojects download

In [ ]:
import pandas as pd

df = pd.read_parquet("full_data.parquet")
df.columns

df_indications = df[["Drug Disease", "Drug Disease Group"]].drop_duplicates()
df_indications.to_excel("all_indications_groups.xlsx", index = False)

Ask Claude to filter for + categorize pain-related indications

In [ ]:
# General rule?
# Exclude: anti-infective, nan, antiparasitic, dermatological, respiratory, NA/Unspecified, Unspecified, nan

df_indications["Drug Disease Group"].unique()

# 

<ArrowStringArray>
[                        'Anti-infective',
                           'Neurological',
                   'Alimentary/Metabolic',
                                      nan,
 'Genitourinary (including sex hormones)',
                         'Cardiovascular',
                             'Anticancer',
                     'Blood and Clotting',
      'Hormonal (excluding sex hormones)',
                          'Antiparasitic',
                         'Dermatological',
                        'Musculoskeletal',
                          'Immunological',
                            'Respiratory',
                                'Sensory',
                         'NA/Unspecified',
                          'Miscellaneous',
                            'Unspecified']
Length: 18, dtype: str

In [25]:
df_pain_indications = pd.read_excel("pain_indications_Claude.xlsx", sheet_name="Pain-related indications")
pain_indications_list = df_pain_indications["Indication"].to_list()

Ask Claude to determine if assets are pain-treating based on MOA


In [ ]:
pain_categories = [
        "Nociceptive / acute & post-operative",
        "Cancer-related",
        "Musculoskeletal",
        "Neuropathic",
        "Nociplastic / centralised",
    ]
indications_to_check = df_pain_indications[(df_pain_indications["Pain category"].isin(pain_categories)) &
                                           (df_pain_indications["Pain relationship"] == "Pain-prominent condition")]["Indication"].to_list()

df_possible_pain_assets = df[df["Drug Disease"].isin(pain_indications_list)]
df_assets_to_check = df_possible_pain_assets[df_possible_pain_assets["Drug Disease"].isin(indications_to_check)]
df_assets_sure = df_possible_pain_assets[~df_possible_pain_assets["Drug Disease"].isin(indications_to_check)]

# Assets we need to check if they are in the pain market and verify/find missing targets
df_assets_to_check = df_assets_to_check[[
    "Citeline Drug ID",
    "Generic Drug Name",
    "Drug Names",
    "Company",
    "Drug Disease",
    "Mechanism Of Action",
    "Target"
]].drop_duplicates()

# Assets we are sure in the pain market, but still need to verify/find missing targets
df_assets_sure = df_assets_sure[[
    "Citeline Drug ID",
    "Generic Drug Name",
    "Drug Names",
    "Company",
    "Drug Disease",
    "Mechanism Of Action",
    "Target"
]].drop_duplicates()

df_assets_sure.to_excel("assets_sure.xlsx", index = False)
# df_assets_to_check.to_excel("assets_to_check.xlsx", index = False)

Get Claude-determined pain assets

In [21]:
# Claude and Alphasense both unable to fill in a lot of the missing targets with the prompts I have so far

df_assets_to_check_verified = pd.read_excel("assets_to_check_verified.xlsx")
df_assets_sure_verified = pd.read_excel("assets_sure_verified.xlsx")

df_pain_assets = pd.concat([df_assets_sure_verified, df_assets_to_check_verified])
df_pain_assets

,Citeline Drug ID,Generic Drug Name,Drug Names,Company,Drug Disease,Mechanism Of Action,Target,verified_target,target_reasoning,target_group,in_pain_market,market_reasoning
0,16,"(R)-ketoprofen, Sepracor","(R)-ketoprofen, Sepracor; (S)-ketoprofen, Sepr...",Sumitomo Pharma,"Pain, nociceptive, general",Cyclooxygenase 1 inhibitor,prostaglandin-endoperoxide synthase 1,prostaglandin-endoperoxide synthase 1,Citeline target retained: consistent with Cite...,Eicosanoid pathway,NaN,NaN
1,20,"(R)-ketorolac, Sumitomo Pharma","(R)-ketorolac, Sumitomo Pharma",Sumitomo Pharma,"Pain, nociceptive, general",Cyclooxygenase 1 inhibitor,prostaglandin-endoperoxide synthase 1,prostaglandin-endoperoxide synthase 1,Citeline target retained: consistent with Cite...,Eicosanoid pathway,NaN,NaN
2,20,"(R)-ketorolac, Sumitomo Pharma","(R)-ketorolac, Sumitomo Pharma",Sumitomo Pharma,"Pain, nociceptive, general",Cyclooxygenase 2 inhibitor,prostaglandin-endoperoxide synthase 2,prostaglandin-endoperoxide synthase 2,Citeline target retained: consistent with Cite...,Eicosanoid pathway,NaN,NaN
3,34,"(S)-mexiletine, Celgene","(S)-mexiletine, Celgene; Mexiletine; Namuscla",Bristol-Myers Squibb,"Pain, nociceptive, general",Voltage-gated sodium channel antagonist,sodium voltage-gated channel alpha subunit 5,sodium voltage-gated channel alpha subunit 5,Citeline target retained: consistent with Cite...,Ion channel,NaN,NaN
4,130,117m-Sn-DTPA,117m-Sn-DTPA; 117mSn-DTPA; Sn-117m DTPA; Sn-11...,Bayer,"Pain, cancer",Radioemitter,Unspecified,Not applicable,"Cell, tissue or device-like product with no si...",Not applicable,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3260,562682,AH-008,AH 008; AH-008; AH008; DBPR 168; DBPR-168; DBP...,AnHorn Medicines,Radio/chemotherapy-induced peripheral neuropathy,Unidentified pharmacological activity,Unspecified,Unspecified,No target in Citeline or ChEMBL/Open Targets; ...,Unspecified,False,Radio/chemotherapy-induced peripheral neuropat...
3261,562620,"etoricoxib + betamethasone, Laboratorios Silanes","etoricoxib + betamethasone, Laboratorios Silanes",Laboratorios Silanes,Synovitis,Glucocorticoid receptor modulator,prostaglandin-endoperoxide synthase 2,prostaglandin-endoperoxide synthase 2,Confirmed: Citeline target(s) match curated Ch...,Eicosanoid pathway,True,Synovitis: Analgesic/anti-inflammatory symptom...
3262,561928,"orthopedic injectable therapy, Human Continuum","orthopedic injectable therapy, Human Continuum...",Human Continuum,"Arthritis, osteo",Unidentified pharmacological activity,Unspecified,Unspecified,No target in Citeline or ChEMBL/Open Targets; ...,Unspecified,False,"Arthritis, osteo: Disease-modifying/structure-..."
3263,569663,PKM-011,PKM 011; PKM-011; PKM011,PK Med,"Arthritis, osteo",Unidentified pharmacological activity,Unspecified,Unspecified,No target in Citeline or ChEMBL/Open Targets; ...,Unspecified,False,"Arthritis, osteo: Disease-modifying/structure-..."


Get final list of Citeline drug IDs for pain assets

In [23]:
df_pain_assets["Citeline Drug ID"].drop_duplicates()

0           16
1           20
3           34
4          130
5          165
         ...  
3253    559358
3254    560058
3260    562682
3263    569663
3264    580932
Name: Citeline Drug ID, Length: 9728, dtype: int64